# Structured output attribution — trace every field back to its source

You ask an LLM to pull structured data out of a document — an order form, an
invoice, a contract — and it returns clean JSON. Useful, but flat: the JSON has
lost all connection to *where in the document each value came from*. For a
human reviewer, an audit trail, or a downstream system, you want every field to
carry a pointer back to the exact source text it was read from.

[TokenPath](https://tokenpath.ai) gives you that in one call: serialize the
extraction, send each field's **value** as a span, and get back the exact
source span behind it, with a confidence attached.

The interesting part is **disambiguation**. Real documents repeat values — the
same percentage, the same amount, the same date appears in two places meaning
two different things. A naive `document.find(value)` always grabs the first
occurrence and silently mislabels the rest. Because attribution reads the model
*in context of the whole JSON*, it resolves each field to the occurrence that
field actually came from. This notebook shows exactly that, on an extraction
that is 100% correct — the only question is *where each value points*.

## Setup

You need a TokenPath API key — free at [platform.tokenpath.ai](https://platform.tokenpath.ai)
(10M attributed tokens on signup, no card required). Export it as `TOKENPATH_API_KEY`
before running this notebook.

In [1]:
%pip install -q requests

In [2]:
import os
import re

import requests

API_URL = "https://api.tokenpath.ai"
API_KEY = os.environ.get("TOKENPATH_API_KEY")
assert API_KEY, (
    "Set TOKENPATH_API_KEY first — grab a free key (10M tokens, no card) "
    "at https://platform.tokenpath.ai"
)


def attribute(document, question, answer, spans, **options):
    """Resolve each answer character span to the source span that produced it."""
    response = requests.post(
        f"{API_URL}/v1/attributions",
        headers={"Authorization": f"Bearer {API_KEY}"},
        json={
            "document": document,
            "question": question,
            "answer": answer,
            "spans": spans,
            **options,
        },
        timeout=120,
    )
    response.raise_for_status()
    return response.json()

## Serialize the JSON so you know where every value landed

Attribution works on character spans of the answer string, so serialize the
extraction yourself and record the `[start, end)` range of every leaf value as
you write it. (For strings the range excludes the quotes, so attribution sees
the words, not the punctuation.)

In [3]:
import json


def dumps_with_spans(value, path="$", indent=0, buf=None, spans=None):
    """json.dumps, but also returns {json_path: [start, end)} for every leaf."""
    if buf is None:
        buf, spans = [], {}

    def emit(text):
        buf.append(text)

    if isinstance(value, dict):
        emit("{\n")
        for i, (key, child) in enumerate(value.items()):
            emit(" " * (indent + 2) + json.dumps(key) + ": ")
            dumps_with_spans(child, f"{path}.{key}", indent + 2, buf, spans)
            emit(",\n" if i < len(value) - 1 else "\n")
        emit(" " * indent + "}")
    elif isinstance(value, list):
        emit("[\n")
        for i, child in enumerate(value):
            emit(" " * (indent + 2))
            dumps_with_spans(child, f"{path}[{i}]", indent + 2, buf, spans)
            emit(",\n" if i < len(value) - 1 else "\n")
        emit(" " * indent + "]")
    else:
        text = json.dumps(value)
        start = sum(len(part) for part in buf)
        if isinstance(value, str):
            spans[path] = [start + 1, start + len(text) - 1]
        else:
            spans[path] = [start, start + len(text)]
        emit(text)

    return "".join(buf), spans


demo_json, demo_spans = dumps_with_spans({"seats": "500", "price": "$49"})
print(demo_json)
demo_spans

{
  "seats": "500",
  "price": "$49"
}


{'$.seats': [14, 17], '$.price': [32, 35]}

## The document and its extraction

An enterprise order form, and the JSON a model extracted from it. The
extraction is **exactly correct** — every value is right. Notice the document
repeats values: **`15%`** appears twice (a first-term reference discount *and*
a renewal price cap), and `24` shows up in both `24 months` and `24 hours`.
Those repeats are the whole point.

In [4]:
DOCUMENT = """\
ACME CLOUD — ENTERPRISE ORDER FORM

Customer: Meridian Health Systems
Order date: February 20, 2026.
Service start date: March 17, 2026.

Subscription: 500 seats at $49 per seat per month, billed annually.
The initial term is 24 months from the service start date, and the
agreement then auto-renews for successive 12-month terms.

As a reference customer, Meridian receives a 15% discount on the first
annual term. Thereafter, the annual price increase is capped at 15% per
renewal. A one-time onboarding fee of $12,000 applies.

Either party may cancel with 60 days written notice before a renewal.
Support response time is 24 hours for standard issues.
"""

QUESTION = "Extract the order terms from this order form as JSON."

EXTRACTION = {
    "customer": "Meridian Health Systems",
    "service_start_date": "March 17, 2026",
    "seats": "500",
    "price_per_seat": "$49 per seat per month",
    "initial_term": "24 months",
    "renewal_term": "12-month",
    "reference_discount": "15%",
    "annual_increase_cap": "15%",
    "onboarding_fee": "$12,000",
    "cancellation_notice": "60 days",
    "support_response_time": "24 hours",
}

answer_json, field_spans = dumps_with_spans(EXTRACTION)
print(answer_json)

{
  "customer": "Meridian Health Systems",
  "service_start_date": "March 17, 2026",
  "seats": "500",
  "price_per_seat": "$49 per seat per month",
  "initial_term": "24 months",
  "renewal_term": "12-month",
  "reference_discount": "15%",
  "annual_increase_cap": "15%",
  "onboarding_fee": "$12,000",
  "cancellation_notice": "60 days",
  "support_response_time": "24 hours"
}


## Attribute every field in one call

`spans` is just the value ranges from the serializer. Each field comes back
with the exact source span it was read from and a confidence.

In [5]:
paths = list(field_spans)
result = attribute(DOCUMENT, QUESTION, answer_json, [field_spans[p] for p in paths])

fields = {}
for path, item in zip(paths, result["spans"]):
    source = item["source"]
    fields[path] = {
        "value": item["answer"]["text"],
        "source_text": source["text"] if source else None,
        "source_start": source["start"] if source else None,
        "confidence": source["confidence"] if source else None,
    }

name_width = max(len(p) for p in fields)
for path, info in fields.items():
    conf = f"{info['confidence']:.2f}" if info["confidence"] is not None else "  — "
    print(f"{path[2:]:{name_width}}  conf {conf}  {info['value']!r}")
    print(f"{'':{name_width}}            <- @{info['source_start']} {info['source_text']!r}")

customer                 conf 1.00  'Meridian Health Systems'
                                   <- @46 'Meridian Health Systems'
service_start_date       conf 0.97  'March 17, 2026'
                                   <- @121 'March 17, 2026'
seats                    conf 0.87  '500'
                                   <- @152 '500'
price_per_seat           conf 1.00  '$49 per seat per month'
                                   <- @165 '$49 per seat per month'
initial_term             conf 0.94  '24 months'
                                   <- @226 '24 months'
renewal_term             conf 1.00  '12-month'
                                   <- @315 '12-month'
reference_discount       conf 0.98  '15%'
                                   <- @377 '15%'
annual_increase_cap      conf 0.63  '15%'
                                   <- @467 '15%'
onboarding_fee           conf 0.98  '$12,000'
                                   <- @513 '$12,000'
cancellation_notice      conf 0.91  '60 days'
      

## The disambiguation, up close

Both `reference_discount` and `annual_increase_cap` have the value `"15%"` —
the *same string*. There are exactly two `15%` in the document. A naive lookup
maps both fields to the first one and gets the cap wrong. Attribution sends
each field to the occurrence it actually came from.

In [6]:
def all_occurrences(needle, haystack):
    out, i = [], haystack.find(needle)
    while i != -1:
        out.append(i)
        i = haystack.find(needle, i + 1)
    return out


print(f"'15%' occurs in the document at offsets: {all_occurrences('15%', DOCUMENT)}\n")

for path in ["$.reference_discount", "$.annual_increase_cap"]:
    info = fields[path]
    naive = all_occurrences(info["value"], DOCUMENT)[0]
    tp = info["source_start"]
    context = DOCUMENT[tp - 40 : tp].replace("\n", " ").strip()
    verdict = "same as naive" if tp == naive else "DIFFERENT from naive — disambiguated"
    print(f"{path[2:]}")
    print(f"  naive str.find -> offset {naive}")
    print(f"  attribution    -> offset {tp}   ({verdict})")
    print(f"  ...{context} [{info['value']}]\n")

'15%' occurs in the document at offsets: [377, 467]

reference_discount
  naive str.find -> offset 377
  attribution    -> offset 377   (same as naive)
  ...reference customer, Meridian receives a [15%]

annual_increase_cap
  naive str.find -> offset 377
  attribution    -> offset 467   (DIFFERENT from naive — disambiguated)
  ...the annual price increase is capped at [15%]



`reference_discount` resolves to the first `15%` (the discount), and
`annual_increase_cap` resolves to the second (the renewal cap) — from the field
name and surrounding JSON alone. The cap's confidence is lower than the
discount's: it's the harder call, and the confidence rides along to tell you
so. We don't threshold on it or drop the field — the span is still the best one
available, and it's the right one. Confidence is a signal you surface, not a
gate you apply.

## Render it: the document with every field highlighted at its source

Each extracted value painted onto the source text at the exact span it came
from — the two `15%` land in different places. This is the audit view you'd put
in front of a reviewer: click a field, see precisely where in the document it
was read.

In [7]:
import html as html_lib

from IPython.display import HTML, display

TINT = "#e6dcf7"


def render_provenance(document, fields):
    spans = sorted(
        (
            (info["source_start"], info["source_start"] + len(info["source_text"]),
             path[2:])
            for path, info in fields.items()
            if info["source_start"] is not None
        ),
        key=lambda s: s[0],
    )
    parts, cursor = [], 0
    for start, end, label in spans:
        if start < cursor:  # this simple renderer skips overlapping spans
            continue
        parts.append(html_lib.escape(document[cursor:start]))
        parts.append(
            f'<span style="background:{TINT};color:#1a1a1a;border-radius:3px;'
            f'padding:0 2px">{html_lib.escape(document[start:end])}'
            f'<sub style="opacity:.6;font-size:9px"> {html_lib.escape(label)}</sub>'
            f"</span>"
        )
        cursor = end
    parts.append(html_lib.escape(document[cursor:]))
    return (
        '<pre style="font-family:Menlo,Consolas,monospace;font-size:12.5px;'
        'line-height:2.1;white-space:pre-wrap;max-width:760px">'
        + "".join(parts)
        + "</pre>"
    )


display(HTML(render_provenance(DOCUMENT, fields)))

## Notes for production extraction pipelines

- **This composes with any extractor** — function calling, JSON mode, a
  fine-tune, an OCR+LLM stack. You attribute the output; the pipeline doesn't
  change. Send attribution the same document you gave the extractor.
- **Provenance, not a verdict.** Attribution tells you *where* each value was
  read from — the receipt a reviewer or auditor needs. It doesn't judge whether
  the extraction is correct; pair it with your own review when you need a
  judgment.
- **Confidence rides along; don't gate on it.** It's a relative signal — lower
  on the harder disambiguations, as you saw on the renewal cap — and it's most
  useful shown next to each field in a review UI, not used as a pass/fail
  threshold.
- **Bare numbers attribute more weakly than distinctive phrases**, so a lone
  reformatted integer can carry lower confidence even when its span is right.
  Give attribution the value in the form it appears in the document when you
  can (`"$49 per seat per month"` beats `49`), and the source span stays exact.
- **Long documents**: the `document` cap is 400k characters — a full contract
  is one call. Past that, attribute per section and merge.

---

## Where to go next

- **API reference & guides** — [docs.tokenpath.ai](https://docs.tokenpath.ai)
- **Bugs / broken examples** — [open an issue](https://github.com/tokenpath/tokenpath-cookbook/issues)
- **"How do I…?"** — [start a discussion](https://github.com/tokenpath/tokenpath-cookbook/discussions)
- **Email** — support@tokenpath.ai